# VeggieLens 蔬菜识别模型训练（第二版）

用 **MediaPipe Model Maker** 在两份 Kaggle 数据集（共 25 类）上做迁移学习，产出 App 可直接加载的 `.tflite` 模型。

**这个笔记本会做什么**

1. 合并两份数据集，类名统一成 App 数据库里的英文名（如 `Bitter_Gourd`、`Onion_Red`）
2. 可选：加入“其他”（Other）类，减少对非蔬菜画面的误判
3. 检查训练集与验证/测试集之间的重复图片（避免测试准确率虚高）
4. 训练、导出浮点版和 **int8 量化版**（体积约为浮点版的 1/4）
5. 用导出的 `.tflite` 逐类评估，画混淆矩阵，找出最容易混淆的蔬菜
6. 按“误判率”标定 App 里的三个阈值，直接输出可粘贴的 Kotlin 代码

**运行环境：本地 WSL2（Ubuntu）+ NVIDIA GPU**

- Model Maker 依赖 TensorFlow < 2.16，需要 **Python 3.9–3.11**
- 必须在 **WSL2** 里运行：Model Maker 依赖的 `tensorflow-text` 没有 Windows 版，Windows 原生的 TensorFlow 也不支持 GPU
- 使用 WSL 里的 conda 环境 **tf216**（名字叫 tf216，实际装的是 TensorFlow 2.15.1 + Model Maker 0.2.1.4），用它的 `jupyter lab` 打开本笔记本，详见 `ml/README.md`
- 数据放在 WSL 的 `~/Data/vegetables/`，产物输出到 `ml/veggielens_export/`

## 步骤 0：环境

In [9]:
import sys

print(sys.version)
assert sys.platform.startswith("linux"), (
    "请在 WSL2（Ubuntu）里运行：Model Maker 在 Windows 上装不上，Windows 原生的 TensorFlow 也不支持 GPU。见 ml/README.md"
)
assert (3, 9) <= sys.version_info[:2] <= (3, 11), (
    "mediapipe-model-maker 依赖 TensorFlow < 2.16，需要 Python 3.9–3.11。请用 conda 环境 tf216 运行（见 ml/README.md）"
)

3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]


In [10]:
# 依赖都装在 conda 环境 tf216 里，这里确认当前内核用的是那个环境
import importlib.util

missing = [m for m in ["mediapipe_model_maker", "imagehash"] if importlib.util.find_spec(m) is None]
assert not missing, (
    f"当前环境缺少 {missing}：请用 conda 环境 tf216 运行；缺 imagehash 就在 tf216 里运行 pip install imagehash"
)

In [11]:
import tensorflow as tf

print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
# 显存按需占用：笔记本显卡同时负责显示，TensorFlow 默认会一次占满
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("GPU:", gpus or "未检测到 GPU，训练会慢很多（确认 WSL 里能运行 nvidia-smi）")

TensorFlow 2.15.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 步骤 1：配置

数据放在 WSL 自己的文件系统里（比放在 `/mnt/d` 下读写快得多）：

```
~/Data/vegetables/
├── Vegetable_Images/       已合并好的 25 类，含 train / validation / test 三个子目录
├── other/                  （建议）非蔬菜图片
├── real_photos/            （建议）手机实拍测试集
└── veggielens_build/       步骤 2 自动生成的合并数据，每次运行都会重建
```

- 类文件夹名写 `Onion Red` 或 `Onion_Red` 都行，步骤 2 会统一成 App 用的类名。原始数据只读不改，去重等操作都在 `veggielens_build/` 的副本上做。
- `other/`：水果、手、桌面、厨房背景、包装袋、超市货架等，**不用分子目录**，脚本按 70/15/15 自动划分。
- `real_photos/`：自己用手机拍的照片，每类一个文件夹，文件夹名用英文类名（如 `Sweet_Potato`），**只用于测试**。也可以放一个 `Other/` 文件夹（真实背景照片），阈值标定会更可靠。
- `other/`、`real_photos/` 不存在或没有图片时自动当作未提供。

In [12]:
from pathlib import Path

# 笔记本所在的 ml/ 目录（Jupyter 的工作目录通常就是笔记本所在目录）
ML_DIR = Path.cwd() if (Path.cwd() / "train_veggielens.ipynb").exists() else Path.cwd() / "ml"
# 数据放在 WSL 的文件系统里，读写比 /mnt/d 快得多
DATA_HOME = Path.home() / "Data" / "vegetables"

DATA_SOURCES = [
    {
        "name": "vegetables",  # 已合并好的 25 类
        "root": DATA_HOME / "Vegetable_Images",
        "splits": {"train": "train", "validation": "validation", "test": "test"},
    },
]

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def has_images(folder: Path) -> bool:
    return folder.is_dir() and any(p.suffix.lower() in IMG_EXT for p in folder.rglob("*") if p.is_file())


# 文件夹里有图片才启用；也可以直接改成别的路径，或写 None
OTHER_DIR = DATA_HOME / "other" if has_images(DATA_HOME / "other") else None
REAL_PHOTOS_DIR = DATA_HOME / "real_photos" if has_images(DATA_HOME / "real_photos") else None

WORK_DIR = DATA_HOME / "veggielens_build"  # 合并后的数据，每次运行步骤 2 都会重建
EXPORT_DIR = ML_DIR / "veggielens_export"  # 模型和报告
APP_ASSETS_DIR = ML_DIR.parent / "app" / "src" / "main" / "assets"

MODEL_VERSION = "v2"
MODEL_SPEC = "EFFICIENTNET_LITE0"  # 可选 EFFICIENTNET_LITE2：更准，但更大更慢（输入 260×260）
EPOCHS = 15
BATCH_SIZE = 32                    # 显存不够（OOM）时改成 16
LEARNING_RATE = 0.0005
FINE_TUNE = True                   # 在同一次训练里再做一次微调，准确率通常明显更高
SEED = 42

# App 中对实拍照片的预处理：取画面中心正方形，边长 = 短边 × 0.8（与 RecognitionConfig.CAMERA_CROP_FRACTION 一致）
CAMERA_CROP_FRACTION = 0.8

# App 数据库中的 25 种蔬菜（模型标签必须与这里一致）
APP_CLASSES = [
    "Bean", "Bitter_Gourd", "Bottle_Gourd", "Brinjal", "Broccoli", "Cabbage", "Capsicum",
    "Carrot", "Cauliflower", "Corn", "Cucumber", "Eggplant", "Galangal", "Garlic", "Ginger",
    "Lettuce", "Onion", "Onion_Red", "Papaya", "Potato", "Pumpkin", "Radish", "Spinach",
    "Sweet_Potato", "Tomato",
]
OTHER_CLASS = "Other"

In [13]:
# 在每个数据源下找到同时含有 train / validation / test 子目录的那一层（取最浅的）
def find_split_root(base: Path, splits: dict) -> Path:
    if base.is_dir():
        dirs = sorted((p for p in [base, *base.rglob("*")] if p.is_dir()), key=lambda p: (len(p.parts), str(p)))
        for d in dirs:
            if all((d / sub).is_dir() for sub in splits.values()):
                return d
    raise FileNotFoundError(
        f"在 {base} 下没找到同时包含 {sorted(splits.values())} 的目录，请确认数据集已解压到这里"
    )


for src in DATA_SOURCES:
    src["split_root"] = find_split_root(Path(src["root"]), src["splits"])
    print(f"== {src['name']}: {src['split_root']}")
    for sub in src["splits"].values():
        classes = sorted(p.name for p in (src["split_root"] / sub).iterdir() if p.is_dir())
        print(f"   {sub}: {len(classes)} 类 {classes}")

print("其他类：", OTHER_DIR or "未提供")
print("手机实拍测试集：", REAL_PHOTOS_DIR or "未提供")

== vegetables: /home/icey/Data/vegetables/Vegetable_Images
   train: 25 类 ['Bean', 'Bitter_Gourd', 'Bottle_Gourd', 'Brinjal', 'Broccoli', 'Cabbage', 'Capsicum', 'Carrot', 'Cauliflower', 'Corn', 'Cucumber', 'Eggplant', 'Galangal', 'Garlic', 'Ginger', 'Lettuce', 'Onion', 'Onion Red', 'Papaya', 'Potato', 'Pumpkin', 'Radish', 'Spinach', 'Sweet Potato', 'Tomato']
   validation: 25 类 ['Bean', 'Bitter_Gourd', 'Bottle_Gourd', 'Brinjal', 'Broccoli', 'Cabbage', 'Capsicum', 'Carrot', 'Cauliflower', 'Corn', 'Cucumber', 'Eggplant', 'Galangal', 'Garlic', 'Ginger', 'Lettuce', 'Onion', 'Onion Red', 'Papaya', 'Potato', 'Pumpkin', 'Radish', 'Spinach', 'Sweet Potato', 'Tomato']
   test: 25 类 ['Bean', 'Bitter_Gourd', 'Bottle_Gourd', 'Brinjal', 'Broccoli', 'Cabbage', 'Capsicum', 'Carrot', 'Cauliflower', 'Corn', 'Cucumber', 'Eggplant', 'Galangal', 'Garlic', 'Ginger', 'Lettuce', 'Onion', 'Onion Red', 'Papaya', 'Potato', 'Pumpkin', 'Radish', 'Spinach', 'Sweet Potato', 'Tomato']
其他类： 未提供
手机实拍测试集： 未提供


## 步骤 2：合并数据集

类名统一规则：按空格、下划线、连字符拆词，每个词首字母大写，再用下划线连接。
例如 `Bitter Gourd` → `Bitter_Gourd`，`Sweet potato` → `Sweet_Potato`，`Onion Red` → `Onion_Red`。

In [14]:
import collections
import hashlib
import re
import shutil

import pandas as pd


def canonical(name: str) -> str:
    words = [w for w in re.split(r"[\s_\-]+", name.strip()) if w]
    return "_".join(w.capitalize() for w in words)


CANON_TO_APP = {canonical(c): c for c in APP_CLASSES}
CANON_TO_APP[canonical(OTHER_CLASS)] = OTHER_CLASS


def images_in(folder: Path):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT)


if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

counts = collections.defaultdict(collections.Counter)
unknown_folders = collections.Counter()

for src in DATA_SOURCES:
    for split, sub in src["splits"].items():
        split_dir = src["split_root"] / sub
        assert split_dir.is_dir(), f"找不到 {split_dir}，请检查 DATA_SOURCES"
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            label = CANON_TO_APP.get(canonical(class_dir.name))
            if label is None:
                unknown_folders[f"{src['name']}/{sub}/{class_dir.name}"] += 1
                continue
            out = WORK_DIR / split / label
            out.mkdir(parents=True, exist_ok=True)
            for img in images_in(class_dir):
                shutil.copy2(img, out / f"{src['name']}__{img.name}")
                counts[split][label] += 1

if OTHER_DIR:
    # 按文件名哈希确定性地划分 70/15/15，重复运行结果一致
    for img in sorted(p for p in Path(OTHER_DIR).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT):
        bucket = int(hashlib.md5(img.name.encode()).hexdigest(), 16) % 100
        split = "train" if bucket < 70 else ("validation" if bucket < 85 else "test")
        out = WORK_DIR / split / OTHER_CLASS
        out.mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, out / f"other__{img.name}")
        counts[split][OTHER_CLASS] += 1

if unknown_folders:
    print("⚠️ 以下文件夹不在 APP_CLASSES 中，已跳过：")
    for k in unknown_folders:
        print("   ", k)

table = pd.DataFrame(counts).fillna(0).astype(int).sort_index()
table["total"] = table.sum(axis=1)
display(table)

expected = set(APP_CLASSES) | ({OTHER_CLASS} if OTHER_DIR else set())
for split in ["train", "validation", "test"]:
    missing = expected - set(counts[split])
    assert not missing, f"{split} 缺少类别：{sorted(missing)}"
print("✅ 三个划分都包含全部类别")

,train,validation,test,total
Bean,1000,200,200,1400
Bitter_Gourd,1000,200,200,1400
Bottle_Gourd,1000,200,200,1400
Brinjal,1000,200,200,1400
Broccoli,1000,200,200,1400
Cabbage,1000,200,200,1400
Capsicum,1000,200,200,1400
Carrot,1000,200,200,1400
Cauliflower,1000,200,200,1400
Corn,979,211,210,1400


✅ 三个划分都包含全部类别


## 步骤 3（建议）：检查重复图片

公开数据集里常有同一张图出现在训练集和测试集，会让测试准确率虚高。这里用感知哈希找出 **与训练集完全相同** 的验证/测试图片，并把它们移除。

In [15]:
import imagehash
from PIL import Image

REMOVE_DUPLICATES = True


def phash(path: Path) -> str:
    with Image.open(path) as im:
        return str(imagehash.phash(im.convert("RGB")))


train_hashes = {phash(p) for p in (WORK_DIR / "train").rglob("*") if p.is_file()}
duplicates = [
    (split, p)
    for split in ["validation", "test"]
    for p in (WORK_DIR / split).rglob("*")
    if p.is_file() and phash(p) in train_hashes
]
print(f"与训练集重复的图片：{len(duplicates)} 张")
for split, n in collections.Counter(s for s, _ in duplicates).items():
    print(f"   {split}: {n}")

if REMOVE_DUPLICATES:
    for _, p in duplicates:
        p.unlink()
    print("已移除。")

/home/icey/miniconda3/envs/tf216/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


与训练集重复的图片：897 张
   validation: 453
   test: 444
已移除。


## 步骤 4：训练

In [16]:
from mediapipe_model_maker import image_classifier, quantization

train_data = image_classifier.Dataset.from_folder(str(WORK_DIR / "train"))
validation_data = image_classifier.Dataset.from_folder(str(WORK_DIR / "validation"))
test_data = image_classifier.Dataset.from_folder(str(WORK_DIR / "test"), shuffle=False)

assert train_data.label_names == validation_data.label_names == test_data.label_names
print("类别数：", train_data.num_classes)
print("标签：", train_data.label_names)
print("训练 / 验证 / 测试：", len(train_data), len(validation_data), len(test_data))

INFO:tensorflow:Load image with size: 24787, num_label: 25, labels: Bean, Bitter_Gourd, Bottle_Gourd, Brinjal, Broccoli, Cabbage, Capsicum, Carrot, Cauliflower, Corn, Cucumber, Eggplant, Galangal, Garlic, Ginger, Lettuce, Onion, Onion_Red, Papaya, Potato, Pumpkin, Radish, Spinach, Sweet_Potato, Tomato.


INFO:tensorflow:Load image with size: 24787, num_label: 25, labels: Bean, Bitter_Gourd, Bottle_Gourd, Brinjal, Broccoli, Cabbage, Capsicum, Carrot, Cauliflower, Corn, Cucumber, Eggplant, Galangal, Garlic, Ginger, Lettuce, Onion, Onion_Red, Papaya, Potato, Pumpkin, Radish, Spinach, Sweet_Potato, Tomato.


INFO:tensorflow:Load image with size: 4657, num_label: 25, labels: Bean, Bitter_Gourd, Bottle_Gourd, Brinjal, Broccoli, Cabbage, Capsicum, Carrot, Cauliflower, Corn, Cucumber, Eggplant, Galangal, Garlic, Ginger, Lettuce, Onion, Onion_Red, Papaya, Potato, Pumpkin, Radish, Spinach, Sweet_Potato, Tomato.


INFO:tensorflow:Load image with size: 4657, num_label: 25, labels: Bean, Bitter_Gourd, Bottle_Gourd, Brinjal, Broccoli, Cabbage, Capsicum, Carrot, Cauliflower, Corn, Cucumber, Eggplant, Galangal, Garlic, Ginger, Lettuce, Onion, Onion_Red, Papaya, Potato, Pumpkin, Radish, Spinach, Sweet_Potato, Tomato.


INFO:tensorflow:Load image with size: 4656, num_label: 25, labels: Bean, Bitter_Gourd, Bottle_Gourd, Brinjal, Broccoli, Cabbage, Capsicum, Carrot, Cauliflower, Corn, Cucumber, Eggplant, Galangal, Garlic, Ginger, Lettuce, Onion, Onion_Red, Papaya, Potato, Pumpkin, Radish, Spinach, Sweet_Potato, Tomato.


INFO:tensorflow:Load image with size: 4656, num_label: 25, labels: Bean, Bitter_Gourd, Bottle_Gourd, Brinjal, Broccoli, Cabbage, Capsicum, Carrot, Cauliflower, Corn, Cucumber, Eggplant, Galangal, Garlic, Ginger, Lettuce, Onion, Onion_Red, Papaya, Potato, Pumpkin, Radish, Spinach, Sweet_Potato, Tomato.


类别数： 25
标签： ['Bean', 'Bitter_Gourd', 'Bottle_Gourd', 'Brinjal', 'Broccoli', 'Cabbage', 'Capsicum', 'Carrot', 'Cauliflower', 'Corn', 'Cucumber', 'Eggplant', 'Galangal', 'Garlic', 'Ginger', 'Lettuce', 'Onion', 'Onion_Red', 'Papaya', 'Potato', 'Pumpkin', 'Radish', 'Spinach', 'Sweet_Potato', 'Tomato']
训练 / 验证 / 测试： 24787 4657 4656


In [17]:
# Model Maker 会从 export_dir/checkpoint 续训，换参数重训前先清空，避免加载旧权重
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

tf.keras.utils.set_random_seed(SEED)

# EfficientNet-Lite 在 TF Hub 上是 TF1 格式，Model Maker 图像分类器用的 hub.KerasLayer 不支持对它微调
# （FINE_TUNE = True 时报 "Setting hub.KerasLayer.trainable = True is unsupported when loading from the TF1 Hub format"）。
# 换成 Model Maker 自带的 HubKerasLayerV1V2：它能微调 TF1 模块，但不更新 BatchNorm 的滑动均值/方差，
# 所以训练时 "Gradients do not exist for variables [...moving_mean...]" 的警告可以忽略。
import types

from mediapipe_model_maker.python.core.utils import hub_loader
from mediapipe_model_maker.python.vision.image_classifier import image_classifier as image_classifier_lib

image_classifier_lib.hub = types.SimpleNamespace(KerasLayer=hub_loader.HubKerasLayerV1V2)

hparams = image_classifier.HParams(
    export_dir=str(EXPORT_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    do_fine_tuning=FINE_TUNE,
    do_data_augmentation=True,  # 随机裁剪、翻转等
    label_smoothing=0.1,
    shuffle=True,
)
options = image_classifier.ImageClassifierOptions(
    supported_model=getattr(image_classifier.SupportedModels, MODEL_SPEC),
    hparams=hparams,
    model_options=image_classifier.ModelOptions(dropout_rate=0.2),
)

model = image_classifier.ImageClassifier.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options,
)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hub_keras_layer_v1v2 (HubK  (None, 1280)              3413024   
 erasLayerV1V2)                                                  
                                                                 
 dropout_1 (Dropout)         (None, 1280)              0         
                                                                 
 dense_1 (Dense)             (None, 25)                32025     
                                                                 
Total params: 3445049 (13.14 MB)
Trainable params: 3445049 (13.14 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None
INFO:tensorflow:Training the models...


INFO:tensorflow:Training the models...


INFO:tensorflow:Use default resize_bicubic.


INFO:tensorflow:Use default resize_bicubic.


Instructions for updating:
Use `tf.image.resize(...method=ResizeMethod.BICUBIC...)` instead.


Instructions for updating:
Use `tf.image.resize(...method=ResizeMethod.BICUBIC...)` instead.


... (213 outputs left out)

2026-10-31 19:48:35.428810: W tensorflow/core/lib/png/png_io.cc:88] PNG warning: iCCP: known incorrect sRGB profile


774/774 [==============================] - ETA: 0s - loss: 0.7200 - accuracy: 0.9778

2026-10-31 19:48:47.018514: W tensorflow/core/lib/png/png_io.cc:88] PNG warning: iCCP: known incorrect sRGB profile
2026-10-31 19:48:47.251098: W tensorflow/core/lib/png/png_io.cc:88] PNG warning: iCCP: known incorrect sRGB profile


774/774 [==============================] - 105s 135ms/step - loss: 0.7200 - accuracy: 0.9778 - val_loss: 0.6802 - val_accuracy: 0.9845


In [18]:
metrics = model.evaluate(test_data, batch_size=BATCH_SIZE)
print("测试集（Keras 模型）：loss = %.4f, accuracy = %.4f" % (metrics[0], metrics[1]))

INFO:tensorflow:Use customized resize method bilinear


INFO:tensorflow:Use customized resize method bilinear


 52/146 [=========>....................] - ETA: 4s - loss: 0.6642 - accuracy: 0.9934

2026-10-31 19:49:08.404930: W tensorflow/core/lib/png/png_io.cc:88] PNG warning: iCCP: known incorrect sRGB profile


120/146 [=======================>......] - ETA: 1s - loss: 0.6536 - accuracy: 0.9964

2026-10-31 19:49:10.899323: W tensorflow/core/lib/png/png_io.cc:88] PNG warning: iCCP: known incorrect sRGB profile


146/146 [==============================] - 7s 45ms/step - loss: 0.6692 - accuracy: 0.9886
测试集（Keras 模型）：loss = 0.6692, accuracy = 0.9886


## 步骤 5：导出模型

In [19]:
FLOAT_NAME = f"veggielens_classifier_{MODEL_VERSION}_float.tflite"
INT8_NAME = f"veggielens_classifier_{MODEL_VERSION}.tflite"

model.export_model(model_name=FLOAT_NAME)

int8_config = quantization.QuantizationConfig.for_int8(representative_data=validation_data)
model.export_model(model_name=INT8_NAME, quantization_config=int8_config)

for name in [FLOAT_NAME, INT8_NAME]:
    path = EXPORT_DIR / name
    print(f"{name}: {path.stat().st_size / 1024 / 1024:.1f} MB")

INFO:tensorflow:Assets written to: /tmp/tmpfb44mg2b/saved_model/assets


INFO:tensorflow:Assets written to: /tmp/tmpfb44mg2b/saved_model/assets
2026-10-31 19:49:26.212219: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-10-31 19:49:26.212396: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-10-31 19:49:26.219170: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpfb44mg2b/saved_model
2026-10-31 19:49:26.238992: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-10-31 19:49:26.239044: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpfb44mg2b/saved_model
2026-10-31 19:49:26.290630: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-10-31 19:49:26.300237: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-10-31 19:49:26.688896: I tensorflow/cc/saved_model/

INFO:tensorflow:TensorFlow Lite model exported successfully to: /mnt/d/Code/Android/VeggieLens/ml/veggielens_export/veggielens_classifier_v2_float.tflite


INFO:tensorflow:TensorFlow Lite model exported successfully to: /mnt/d/Code/Android/VeggieLens/ml/veggielens_export/veggielens_classifier_v2_float.tflite


INFO:tensorflow:Assets written to: /tmp/tmpmh_3m7n_/saved_model/assets


INFO:tensorflow:Assets written to: /tmp/tmpmh_3m7n_/saved_model/assets


INFO:tensorflow:Use customized resize method bilinear


INFO:tensorflow:Use customized resize method bilinear
2026-10-31 19:49:34.429540: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-10-31 19:49:34.429624: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-10-31 19:49:34.429838: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpmh_3m7n_/saved_model
2026-10-31 19:49:34.448005: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-10-31 19:49:34.448056: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpmh_3m7n_/saved_model
2026-10-31 19:49:34.515663: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-10-31 19:49:34.907266: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpmh_3m7n_/saved_model
2026-10-31 19:49:35.118483: I tensorflow/cc/saved_

INFO:tensorflow:TensorFlow Lite model exported successfully to: /mnt/d/Code/Android/VeggieLens/ml/veggielens_export/veggielens_classifier_v2.tflite


INFO:tensorflow:TensorFlow Lite model exported successfully to: /mnt/d/Code/Android/VeggieLens/ml/veggielens_export/veggielens_classifier_v2.tflite


veggielens_classifier_v2_float.tflite: 12.9 MB
veggielens_classifier_v2.tflite: 4.0 MB


## 步骤 6：用导出的 .tflite 逐类评估

这里用 App 里同一个推理库（MediaPipe Tasks）加载导出的模型，结果就是 App 实际会得到的分数。
同时评估浮点版和 int8 版，确认量化没有明显掉点。

In [20]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision


def load_classifier(model_path: Path):
    opts = vision.ImageClassifierOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(model_path)),
        max_results=3,
    )
    return vision.ImageClassifier.create_from_options(opts)


def center_crop(im: Image.Image, fraction: float) -> Image.Image:
    side = int(min(im.size) * fraction)
    left = (im.width - side) // 2
    top = (im.height - side) // 2
    return im.crop((left, top, left + side, top + side))


def predict_folder(classifier, root: Path, crop_fraction=None) -> pd.DataFrame:
    rows = []
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        true_label = CANON_TO_APP.get(canonical(class_dir.name))
        if true_label is None:
            print("跳过未知类别文件夹：", class_dir.name)
            continue
        for path in images_in(class_dir):
            with Image.open(path) as im:
                im = im.convert("RGB")
                if crop_fraction:
                    im = center_crop(im, crop_fraction)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(im))
            categories = classifier.classify(mp_image).classifications[0].categories
            top = categories[0] if categories else None
            rows.append({
                "file": str(path),
                "true": true_label,
                "pred": top.category_name if top else None,
                "score": top.score if top else 0.0,
                "top3": [c.category_name for c in categories],
            })
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame, title: str):
    top1 = (df["pred"] == df["true"]).mean()
    top3 = np.mean([t in p for t, p in zip(df["true"], df["top3"])])
    print(f"{title}：{len(df)} 张，Top-1 准确率 {top1:.2%}，Top-3 准确率 {top3:.2%}")
    return top1


results = {}
for name in [FLOAT_NAME, INT8_NAME]:
    clf = load_classifier(EXPORT_DIR / name)
    results[name] = predict_folder(clf, WORK_DIR / "test")
    summarize(results[name], f"测试集 · {name}")

if REAL_PHOTOS_DIR:
    clf = load_classifier(EXPORT_DIR / INT8_NAME)
    results["real"] = predict_folder(clf, Path(REAL_PHOTOS_DIR), crop_fraction=CAMERA_CROP_FRACTION)
    summarize(results["real"], f"手机实拍 · {INT8_NAME}（已按 App 的方式中心裁剪）")

I0000 00:00:1793447410.285396      42 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1793447410.340376    5849 gl_context.cc:369] GL version: 3.1 (OpenGL ES 3.1 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: D3D12 (NVIDIA GeForce RTX 4060 Laptop GPU)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


测试集 · veggielens_classifier_v2_float.tflite：4656 张，Top-1 准确率 98.35%，Top-3 准确率 99.89%


I0000 00:00:1793447491.701184      42 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1793447491.742847    5869 gl_context.cc:369] GL version: 3.1 (OpenGL ES 3.1 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: D3D12 (NVIDIA GeForce RTX 4060 Laptop GPU)


测试集 · veggielens_classifier_v2.tflite：4656 张，Top-1 准确率 98.52%，Top-3 准确率 99.87%


In [21]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

EVAL_KEY = "real" if "real" in results else INT8_NAME
df = results[EVAL_KEY]
labels = sorted(set(df["true"]) | set(df["pred"].dropna()))

print(f"逐类结果（{EVAL_KEY}）")
print(classification_report(df["true"], df["pred"].fillna("None"), labels=labels, digits=3, zero_division=0))

cm = confusion_matrix(df["true"], df["pred"].fillna("None"), labels=labels)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)), labels, rotation=90)
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalized)")
fig.colorbar(im)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

wrong = df[df["pred"] != df["true"]]
print("最常见的误判（真实 → 预测）：")
display(wrong.groupby(["true", "pred"]).size().sort_values(ascending=False).head(15).rename("次数"))

逐类结果（veggielens_classifier_v2.tflite）
              precision    recall  f1-score   support

        Bean      1.000     1.000     1.000       192
Bitter_Gourd      1.000     0.995     0.997       191
Bottle_Gourd      1.000     1.000     1.000       185
     Brinjal      0.995     1.000     0.997       197
    Broccoli      1.000     1.000     1.000       190
     Cabbage      1.000     0.995     0.997       198
    Capsicum      1.000     1.000     1.000       193
      Carrot      1.000     1.000     1.000       185
 Cauliflower      0.995     0.995     0.995       196
        Corn      0.989     0.994     0.992       180
    Cucumber      1.000     0.995     0.997       187
    Eggplant      0.990     0.995     0.992       198
    Galangal      0.966     0.980     0.973       204
      Garlic      0.938     1.000     0.968       136
      Ginger      0.980     0.975     0.978       202
     Lettuce      0.969     1.000     0.984       126
       Onion      0.967     0.741     0.839

/tmp/ipykernel_42/3047360262.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


true          pred        
Onion         Onion_Red       28
              Garlic           7
Spinach       Lettuce          3
Onion         Ginger           2
Sweet_Potato  Galangal         2
Potato        Sweet_Potato     2
Galangal      Onion_Red        2
Ginger        Galangal         2
              Onion_Red        2
Onion         Galangal         2
Bitter_Gourd  Corn             1
Sweet_Potato  Ginger           1
              Garlic           1
Spinach       Ginger           1
Onion_Red     Onion            1
Name: 次数, dtype: int64

## 步骤 7：标定 App 的阈值

App 里有三个阈值（单位是“匹配度百分比”= 分数 × 100）：

| 阈值 | 用途 | 标定目标 |
| --- | --- | --- |
| `AUTO_SCAN_MIN_CONFIDENCE` | 自动识别直接出结果 | 出结果时准确率 ≥ 97% |
| `MANUAL_SCAN_MIN_CONFIDENCE` | 手动拍照 / 相册识别 | 准确率 ≥ 90% |
| `HINT_MIN_CONFIDENCE` | 取景时提示“像是 xx” | 准确率 ≥ 75% |

“覆盖率”是蔬菜图片中能拿到结果的比例。阈值越高越准，但覆盖率越低。有手机实拍集时优先用它标定。

In [22]:
import json

def calibration_table(df: pd.DataFrame) -> pd.DataFrame:
    veg = df[df["true"] != OTHER_CLASS]
    rows = []
    for t in range(5, 100, 5):
        accepted = df[(df["pred"] != OTHER_CLASS) & (df["score"] * 100 >= t)]
        accepted_veg = accepted[accepted["true"] != OTHER_CLASS]
        rows.append({
            "阈值": t,
            "准确率": (accepted["pred"] == accepted["true"]).mean() if len(accepted) else np.nan,
            "覆盖率": len(accepted_veg) / max(len(veg), 1),
        })
    return pd.DataFrame(rows)


def pick_threshold(table: pd.DataFrame, target: float, fallback: int) -> int:
    ok = table[table["准确率"] >= target]
    return int(ok["阈值"].min()) if len(ok) else fallback


cal = calibration_table(results[EVAL_KEY])
display(cal.round(3))

thresholds = {
    "AUTO_SCAN_MIN_CONFIDENCE": pick_threshold(cal, 0.97, 85),
    "MANUAL_SCAN_MIN_CONFIDENCE": pick_threshold(cal, 0.90, 50),
    "HINT_MIN_CONFIDENCE": pick_threshold(cal, 0.75, 30),
    "ALTERNATIVE_MIN_CONFIDENCE": 15,
}

print("\n把下面几行替换到 app/src/main/java/com/example/veggielens/viewmodel/RecognitionConfig.kt：\n")
print(f'    const val MODEL_ASSET = "{INT8_NAME}"')
for key, value in thresholds.items():
    print(f"    const val {key} = {value}")

report = {
    "model": INT8_NAME,
    "model_spec": MODEL_SPEC,
    "epochs": EPOCHS,
    "labels": train_data.label_names,
    "class_counts": {split: dict(c) for split, c in counts.items()},
    "test_top1_float": float((results[FLOAT_NAME]["pred"] == results[FLOAT_NAME]["true"]).mean()),
    "test_top1_int8": float((results[INT8_NAME]["pred"] == results[INT8_NAME]["true"]).mean()),
    "real_top1": float((results["real"]["pred"] == results["real"]["true"]).mean()) if "real" in results else None,
    "thresholds": thresholds,
}
(EXPORT_DIR / "training_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
print("\n报告已保存：", EXPORT_DIR / "training_report.json")

,阈值,准确率,覆盖率
0,5,0.985,1.000
1,10,0.985,1.000
2,15,0.985,1.000
3,20,0.986,0.999
4,25,0.986,0.999
5,30,0.987,0.997
6,35,0.988,0.995
7,40,0.989,0.994
8,45,0.990,0.991
9,50,0.991,0.984



把下面几行替换到 app/src/main/java/com/example/veggielens/viewmodel/RecognitionConfig.kt：

    const val MODEL_ASSET = "veggielens_classifier_v2.tflite"
    const val AUTO_SCAN_MIN_CONFIDENCE = 5
    const val MANUAL_SCAN_MIN_CONFIDENCE = 5
    const val HINT_MIN_CONFIDENCE = 5
    const val ALTERNATIVE_MIN_CONFIDENCE = 15

报告已保存： /mnt/d/Code/Android/VeggieLens/ml/veggielens_export/training_report.json


## 步骤 8：放进 App

1. 运行下一格，把 int8 模型复制到 `app/src/main/assets/`（只是多放一个文件，改 `MODEL_ASSET` 之前 App 仍加载旧模型）
2. 按上一步输出更新 `RecognitionConfig.kt` 里的 `MODEL_ASSET` 和阈值
3. 运行单元测试：`./gradlew test`（`VegetableMatcherTest` 会检查每个标签都能对应到正确的蔬菜）
4. 真机上对着几种蔬菜试一试，尤其是混淆矩阵里最容易混的几对
5. 确认没问题后，可以删除旧的 `veggielens_classifier_v1.tflite` 以减小安装包

`ml/veggielens_export/` 里的 `confusion_matrix.png` 和 `training_report.json` 建议一起保存，方便和下一版对比。

In [23]:
assert APP_ASSETS_DIR.is_dir(), f"找不到 {APP_ASSETS_DIR}"
shutil.copy2(EXPORT_DIR / INT8_NAME, APP_ASSETS_DIR / INT8_NAME)
print("已复制到", APP_ASSETS_DIR / INT8_NAME)

已复制到 /mnt/d/Code/Android/VeggieLens/app/src/main/assets/veggielens_classifier_v2.tflite
